<a href="https://colab.research.google.com/github/sim07hamdani/FDM5-3P-Travaux/blob/main/notebooks/sample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
from google.colab import drive
# --- 1. Montage de Drive et dossier de travail ---------------
drive.mount("/content/drive")
WORKDIR = "/content/drive/MyDrive/these/run02"
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
print("Dossier de travail :", os.getcwd())

Mounted at /content/drive
Dossier de travail : /content/drive/MyDrive/these/run02


In [4]:
%%writefile test0.py
#!/usr/bin/env python3
"""
Generate normally distributed data, save to CSV, reload it, run a statistical
analysis, and save every figure as a PDF.

Dataset (n rows):
    group_a    ~ N(50, 10^2)
    group_b    ~ N(55, 12^2)
    x          ~ N(0, 1)
    y          = 2.0 + 1.5 * x + eps,  eps ~ N(0, 1)

Analyses:
    - Descriptive statistics with 95% t-based confidence intervals
    - Normality: Shapiro-Wilk, D'Agostino-Pearson K^2, Anderson-Darling
    - Group comparison: Welch t-test, Mann-Whitney U, Levene, Cohen's d
    - Association: Pearson, Spearman, OLS regression (slope CI, R^2, residual check)

Usage:
    python normal_data_analysis.py
    python normal_data_analysis.py --n 2000 --seed 7 --outdir out

Requires: numpy, pandas, scipy, matplotlib
"""

import argparse
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # headless backend, no display required
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

plt.rcParams.update({
    "pdf.fonttype": 42,          # embed TrueType fonts (editable text in PDF)
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 100,
})

CONFIDENCE = 0.95
ALPHA = 1.0 - CONFIDENCE


# --------------------------------------------------------------------------- #
# 1. Data generation
# --------------------------------------------------------------------------- #
def generate_data(n: int, seed: int, csv_path: Path) -> None:
    """Draw synthetic normal data and write it to csv_path."""
    rng = np.random.default_rng(seed)
    x = rng.normal(0.0, 1.0, n)
    df = pd.DataFrame({
        "group_a": rng.normal(50.0, 10.0, n),
        "group_b": rng.normal(55.0, 12.0, n),
        "x": x,
        "y": 2.0 + 1.5 * x + rng.normal(0.0, 1.0, n),
    })
    df.to_csv(csv_path, index=False)


# --------------------------------------------------------------------------- #
# 2. Statistical analysis
# --------------------------------------------------------------------------- #
def describe(s: pd.Series) -> dict:
    n = s.size
    mean, sd = s.mean(), s.std(ddof=1)
    sem = sd / np.sqrt(n)
    tcrit = stats.t.ppf(1 - ALPHA / 2, df=n - 1)
    return {
        "n": n,
        "mean": mean,
        "sd": sd,
        "sem": sem,
        "ci_low": mean - tcrit * sem,
        "ci_high": mean + tcrit * sem,
        "median": s.median(),
        "min": s.min(),
        "max": s.max(),
        "skew": stats.skew(s, bias=False),
        "excess_kurtosis": stats.kurtosis(s, bias=False),  # Fisher, 0 for normal
    }


def normality_tests(s: pd.Series) -> dict:
    out = {}
    w, p = stats.shapiro(s)
    out["shapiro_W"], out["shapiro_p"] = w, p
    if s.size >= 8:  # normaltest requires n >= 8
        k2, p = stats.normaltest(s)
        out["dagostino_K2"], out["dagostino_p"] = k2, p
    try:  # SciPy >= 1.17: p-value from interpolated tables
        ad = stats.anderson(s, dist="norm", method="interpolate")
        out["anderson_A2"], out["anderson_p"] = ad.statistic, ad.pvalue
    except TypeError:  # older SciPy: compare A^2 with the 5% critical value
        ad = stats.anderson(s, dist="norm")
        idx = list(ad.significance_level).index(5.0)
        out["anderson_A2"] = ad.statistic
        out["anderson_crit_5pct"] = ad.critical_values[idx]
    return out


def cohens_d(a: np.ndarray, b: np.ndarray) -> float:
    na, nb = a.size, b.size
    pooled = np.sqrt(((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2))
    return (a.mean() - b.mean()) / pooled


def group_comparison(a: np.ndarray, b: np.ndarray) -> dict:
    t = stats.ttest_ind(a, b, equal_var=False)  # Welch
    ci = t.confidence_interval(CONFIDENCE)
    u = stats.mannwhitneyu(a, b, alternative="two-sided")
    lev = stats.levene(a, b, center="median")  # Brown-Forsythe
    return {
        "welch_t": t.statistic,
        "welch_df": t.df,
        "welch_p": t.pvalue,
        "mean_diff_a_minus_b": a.mean() - b.mean(),
        "mean_diff_ci_low": ci.low,
        "mean_diff_ci_high": ci.high,
        "cohens_d": cohens_d(a, b),
        "mannwhitney_U": u.statistic,
        "mannwhitney_p": u.pvalue,
        "levene_W": lev.statistic,
        "levene_p": lev.pvalue,
    }


def regression(x: np.ndarray, y: np.ndarray) -> dict:
    n = x.size
    res = stats.linregress(x, y)
    tcrit = stats.t.ppf(1 - ALPHA / 2, df=n - 2)
    fitted = res.intercept + res.slope * x
    resid = y - fitted
    s_err = np.sqrt(np.sum(resid ** 2) / (n - 2))
    sw = stats.shapiro(resid)
    return {
        "slope": res.slope,
        "slope_se": res.stderr,
        "slope_ci_low": res.slope - tcrit * res.stderr,
        "slope_ci_high": res.slope + tcrit * res.stderr,
        "intercept": res.intercept,
        "intercept_se": res.intercept_stderr,
        "intercept_ci_low": res.intercept - tcrit * res.intercept_stderr,
        "intercept_ci_high": res.intercept + tcrit * res.intercept_stderr,
        "r": res.rvalue,
        "r_squared": res.rvalue ** 2,
        "slope_p": res.pvalue,
        "residual_se": s_err,
        "residual_shapiro_W": sw.statistic,
        "residual_shapiro_p": sw.pvalue,
        "_fitted": fitted,
        "_resid": resid,
        "_tcrit": tcrit,
    }


# --------------------------------------------------------------------------- #
# 3. Figures (one PDF each)
# --------------------------------------------------------------------------- #
def save(fig: plt.Figure, path: Path) -> Path:
    fig.tight_layout()
    fig.savefig(path, format="pdf", bbox_inches="tight")
    plt.close(fig)
    return path


def fig_histograms(df: pd.DataFrame, path: Path) -> Path:
    fig, axes = plt.subplots(2, 2, figsize=(9, 7))
    for ax, col in zip(axes.ravel(), df.columns):
        s = df[col]
        ax.hist(s, bins="auto", density=True, alpha=0.6, edgecolor="white", label="Data")
        grid = np.linspace(s.min(), s.max(), 400)
        ax.plot(grid, stats.norm.pdf(grid, s.mean(), s.std(ddof=1)), lw=2,
                label=r"Fitted $\mathcal{N}(\bar{x}, s^2)$")
        ax.plot(grid, stats.gaussian_kde(s)(grid), lw=1.5, ls="--", label="KDE")
        ax.axvline(s.mean(), color="k", lw=1, ls=":")
        ax.set_title(col)
        ax.set_xlabel("Value")
        ax.set_ylabel("Density")
        ax.legend(frameon=False, fontsize=8)
    return save(fig, path)


def fig_qq(df: pd.DataFrame, path: Path) -> Path:
    fig, axes = plt.subplots(2, 2, figsize=(9, 7))
    for ax, col in zip(axes.ravel(), df.columns):
        (osm, osr), (slope, intercept, r) = stats.probplot(df[col], dist="norm")
        ax.plot(osm, osr, ".", alpha=0.5)
        ax.plot(osm, slope * osm + intercept, "r-", lw=1.5)
        ax.set_title(f"{col}  (probplot $R^2$ = {r**2:.4f})")
        ax.set_xlabel("Theoretical quantiles")
        ax.set_ylabel("Sample quantiles")
    return save(fig, path)


def fig_group_comparison(df: pd.DataFrame, cmp: dict, path: Path) -> Path:
    fig, ax = plt.subplots(figsize=(5.5, 5))
    data = [df["group_a"], df["group_b"]]
    parts = ax.violinplot(data, showextrema=False)
    for body in parts["bodies"]:
        body.set_alpha(0.35)
    ax.boxplot(data, widths=0.15, showfliers=False, medianprops={"color": "k"})
    means = [d.mean() for d in data]
    ax.plot([1, 2], means, "D", color="crimson", label="Mean")
    ax.set_xticks([1, 2], ["group_a", "group_b"])
    ax.set_ylabel("Value")
    ax.set_title(
        f"Welch t = {cmp['welch_t']:.2f}, p = {cmp['welch_p']:.2e}, "
        f"Cohen's d = {cmp['cohens_d']:.2f}",
        fontsize=9,
    )
    ax.legend(frameon=False)
    return save(fig, path)


def fig_regression(df: pd.DataFrame, reg: dict, path: Path) -> Path:
    x, y = df["x"].to_numpy(), df["y"].to_numpy()
    n = x.size
    grid = np.linspace(x.min(), x.max(), 300)
    yhat = reg["intercept"] + reg["slope"] * grid
    sxx = np.sum((x - x.mean()) ** 2)
    se_mean = reg["residual_se"] * np.sqrt(1 / n + (grid - x.mean()) ** 2 / sxx)
    se_pred = reg["residual_se"] * np.sqrt(1 + 1 / n + (grid - x.mean()) ** 2 / sxx)
    t = reg["_tcrit"]

    fig, ax = plt.subplots(figsize=(6.5, 5))
    ax.scatter(x, y, s=10, alpha=0.4, label="Data")
    ax.plot(grid, yhat, "r-", lw=2,
            label=f"OLS: y = {reg['intercept']:.2f} + {reg['slope']:.2f} x")
    ax.fill_between(grid, yhat - t * se_mean, yhat + t * se_mean, color="r", alpha=0.25,
                    label="95% CI (mean response)")
    ax.fill_between(grid, yhat - t * se_pred, yhat + t * se_pred, color="gray", alpha=0.15,
                    label="95% prediction interval")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(f"$R^2$ = {reg['r_squared']:.3f}, slope p = {reg['slope_p']:.2e}")
    ax.legend(frameon=False, fontsize=8)
    return save(fig, path)


def fig_residuals(reg: dict, path: Path) -> Path:
    fitted, resid = reg["_fitted"], reg["_resid"]
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))

    axes[0].scatter(fitted, resid, s=10, alpha=0.4)
    axes[0].axhline(0, color="r", lw=1)
    axes[0].set(xlabel="Fitted values", ylabel="Residuals", title="Residuals vs fitted")

    stats.probplot(resid, dist="norm", plot=axes[1])
    axes[1].set_title("Normal Q-Q of residuals")

    axes[2].hist(resid, bins="auto", density=True, alpha=0.6, edgecolor="white")
    grid = np.linspace(resid.min(), resid.max(), 300)
    axes[2].plot(grid, stats.norm.pdf(grid, 0, resid.std(ddof=2)), lw=2)
    axes[2].set(xlabel="Residual", ylabel="Density", title="Residual distribution")
    return save(fig, path)


def fig_correlation(df: pd.DataFrame, path: Path) -> Path:
    corr = df.corr(method="pearson")
    fig, ax = plt.subplots(figsize=(5.5, 4.8))
    im = ax.imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_xticks(range(len(corr)), corr.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(corr)), corr.columns)
    for i in range(len(corr)):
        for j in range(len(corr)):
            ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=9)
    fig.colorbar(im, ax=ax, label="Pearson r")
    ax.set_title("Correlation matrix")
    ax.spines[:].set_visible(False)
    return save(fig, path)


# --------------------------------------------------------------------------- #
# 4. Reporting
# --------------------------------------------------------------------------- #
def fmt(v) -> str:
    if isinstance(v, (int, np.integer)):
        return f"{v:d}"
    if isinstance(v, (float, np.floating)):
        return f"{v:.4g}" if abs(v) >= 1e-3 or v == 0 else f"{v:.3e}"
    return str(v)


def section(title: str, d: dict) -> list[str]:
    lines = [f"\n== {title} =="]
    lines += [f"  {k:<22s} {fmt(v)}" for k, v in d.items() if not k.startswith("_")]
    return lines


# --------------------------------------------------------------------------- #
# 5. Main
# --------------------------------------------------------------------------- #
def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__,
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--n", type=int, default=1000, help="rows to generate (default 1000)")
    ap.add_argument("--seed", type=int, default=42, help="RNG seed (default 42)")
    ap.add_argument("--outdir", type=Path, default=Path("output"), help="output directory")
    args = ap.parse_args()
    if args.n < 8:
        ap.error("--n must be >= 8 for the normality tests")

    args.outdir.mkdir(parents=True, exist_ok=True)
    csv_path = args.outdir / "normal_data.csv"

    # Generate, write, then read back from disk so the analysis uses the CSV.
    generate_data(args.n, args.seed, csv_path)
    df = pd.read_csv(csv_path)

    report = [f"Dataset: {csv_path}  (n = {len(df)}, seed = {args.seed})"]

    # Descriptive statistics and normality, per column
    desc = pd.DataFrame({c: describe(df[c]) for c in df.columns}).T
    report.append("\n== Descriptive statistics (95% t-based CI on the mean) ==")
    report.append(desc.to_string(float_format=lambda v: f"{v:.4g}"))

    norm = pd.DataFrame({c: normality_tests(df[c]) for c in df.columns}).T
    report.append("\n== Normality tests (H0: sample drawn from a normal distribution) ==")
    report.append(norm.to_string(float_format=lambda v: f"{v:.4g}"))

    # Inference
    cmp = group_comparison(df["group_a"].to_numpy(), df["group_b"].to_numpy())
    report += section("Group comparison: group_a vs group_b", cmp)

    pr = stats.pearsonr(df["x"], df["y"])
    sr = stats.spearmanr(df["x"], df["y"])
    report += section("Association: x vs y", {
        "pearson_r": pr.statistic,
        "pearson_p": pr.pvalue,
        "pearson_ci_low": pr.confidence_interval(CONFIDENCE).low,
        "pearson_ci_high": pr.confidence_interval(CONFIDENCE).high,
        "spearman_rho": sr.statistic,
        "spearman_p": sr.pvalue,
    })

    reg = regression(df["x"].to_numpy(), df["y"].to_numpy())
    report += section("OLS regression: y ~ x", reg)

    # Figures
    figs = [
        fig_histograms(df, args.outdir / "fig1_histograms.pdf"),
        fig_qq(df, args.outdir / "fig2_qq_plots.pdf"),
        fig_group_comparison(df, cmp, args.outdir / "fig3_group_comparison.pdf"),
        fig_regression(df, reg, args.outdir / "fig4_regression.pdf"),
        fig_residuals(reg, args.outdir / "fig5_residual_diagnostics.pdf"),
        fig_correlation(df, args.outdir / "fig6_correlation_matrix.pdf"),
    ]

    report.append("\n== Files written ==")
    report += [f"  {p}" for p in [csv_path, *figs]]

    text = "\n".join(report)
    (args.outdir / "results.txt").write_text(text + "\n")
    print(text)


if __name__ == "__main__":
    main()

Overwriting test0.py


In [5]:
%run test0.py

Dataset: output/normal_data.csv  (n = 1000, seed = 42)

== Descriptive statistics (95% t-based CI on the mean) ==
           n     mean     sd     sem   ci_low  ci_high   median    min   max     skew  excess_kurtosis
group_a 1000    49.19  10.15  0.3208    48.56    49.82    49.56  19.52 79.14 -0.04797         -0.07198
group_b 1000     55.4   12.2  0.3858    54.65    56.16    55.44  18.23 91.71 -0.01685         -0.02852
x       1000 -0.02889 0.9892 0.03128 -0.09028  0.03249 0.006178 -3.648 3.179 -0.04375          0.09188
y       1000    1.955  1.783 0.05637    1.845    2.066    1.936 -4.366 8.343  0.02673           0.2166

== Normality tests (H0: sample drawn from a normal distribution) ==
         shapiro_W  shapiro_p  dagostino_K2  dagostino_p  anderson_A2  anderson_crit_5pct
group_a     0.9987     0.7018        0.5573       0.7568       0.2804               0.784
group_b     0.9988     0.7847       0.06083         0.97       0.2058               0.784
x           0.9988     0.7576   